In [19]:
from pathlib import Path

import automech
import polars as pl
from automech.species import Species
from project_utilities import p_, util

file = util.notebook_file() if util.is_notebook() else __file__
root_path = Path("..")
chemkin_path = p_.chemkin(root_path)

In [20]:
def select_cho_species(spc_df: pl.DataFrame) -> pl.DataFrame:
    all_symbols = {f.name for f in spc_df.schema["formula"].fields}  # ty:ignore[unresolved-attribute]
    included_symbols = {"C", "H", "O"}
    excluded_symbols = all_symbols - included_symbols

    return spc_df.filter(
        pl.all_horizontal(
            [
                pl.col(Species.formula).struct.field(s).fill_null(0) == 0
                for s in excluded_symbols
            ]
        )
    )


def racemize(spc_df: pl.DataFrame) -> pl.DataFrame:
    return spc_df.with_columns(pl.col(Species.name).str.replace(r"[01]$", "R")).unique(
        subset=Species.name
    )


failed_species = [
    "C5H7(1315)",
    "S(1402)rR",
    "S(1336)rR",
    "S(1641)ze",
    "S(1633)",
    "S(1629)",
    "S(1641)ez",
    "S(1406)rR",
    "S(1250)e",
    "C5H8O(837)z",
    "S(1562)rsR",
    "S(1445)z",
    "S(1421)ze",
    "S(1511)e",
    "S(1492)rsR",
    "S(1583)z",
    "S(1344)z",
    "S(163)zz",
    "S(1411)rR",
    "S(1485)e",
    "S(1335)rR",
    "S(1477)",
    "S(1615)",
    "C5H9O(856)",
    "S(1344)e",
    "S(1644)rsR",
    "S(1641)ee",
    "S(1445)e",
    "S(1673)zz",
    "S(1635)rR",
    "S(1492)rrR",
    "S(1673)ez",
    "S(1370)",
    "S(1388)",
    "S(1588)z",
    "S(1348)rR",
    "S(1502)eeee",
    "S(1613)z",
    "C4H7(1434)",
    "S(1373)rsR",
    "S(1579)rR",
    "S(1644)rrR",
    "S(1588)e",
    "S(1453)eee",
    "S(1613)e",
    "S(1599)",
    "S(1583)e",
    "S(1624)",
    "S(1374)rR",
    "S(1640)zz",
    "S(1641)zz",
    "S(1250)z",
    "S(1532)erR",
    "S(1504)rsR",
    "S(1385)z",
    "S(1562)rrR",
    "S(1500)rsR",
    "S(1569)",
]


def drop_failed_species(spc_df: pl.DataFrame) -> pl.DataFrame:
    return spc_df.filter(~pl.col(Species.name).is_in(failed_species))

In [21]:
tag0 = "cyclopentene_species_expanded"

# 1a. Read in ASC species set
spc_file0 = p_.data(root_path) / f"{tag0}.json"
spc_df0 = pl.read_json(spc_file0)
spc_df0 = select_cho_species(spc_df0)
spc_df0 = racemize(spc_df0)
spc_df0 = drop_failed_species(spc_df0)

# 2a. Read in ASC thermochemistry
therm_file0 = p_.chemkin(root_path) / f"{tag0}.dat"
therm_file0

spc_df0 = automech.io.chemkin.read.thermo(
    therm_file0, spc_df=spc_df0, racemize=True, complete=True
)
spc_df0

name,smiles,amchi,spin,charge,formula,canon,orig_name,orig_smiles,orig_amchi,therm
str,str,str,i64,i64,struct[18],bool,str,str,str,struct[8]
"""S(1481)""","""C=C1CO1""","""AMChI=1/C3H4O/c1-3-2-4-3/h1-2H…",0,0,"{4,null,null,null,null,3,null,1,null,null,null,null,null,null,null,null,null,null}",true,"""S(1481)""","""C=C1OC1""","""AMChI=1/C3H4O/c1-3-2-4-3/h1-2H…","{200.0,3000.0,{3,4,1},0,1000.0,[2.043575, 0.013631, … 15.800858],[4.578369, 0.019336, … 0.180155],""nasa7""}"
"""S(1256)eee""","""[CH2]/C=/C/C=/C/O""","""AMChI=1/C5H7O/c1-2-3-4-5-6/h2-…",1,0,"{7,null,null,null,null,5,null,1,null,null,null,null,null,null,null,null,null,null}",true,"""S(1256)""","""OC=CC=C[CH2]""","""AMChI=1/C5H7O/c1-2-3-4-5-6/h2-…","{200.0,3000.0,{5,7,1},0,1000.0,[1.912188, 0.047523, … 17.177422],[8.9237599, 0.0296, … -18.910215],""nasa7""}"
"""S(1257)z""","""C=CC/C=[C]\O""","""AMChI=1/C5H7O/c1-3-5-4-2-6/h3-…",1,0,"{7,null,null,null,null,5,null,1,null,null,null,null,null,null,null,null,null,null}",true,"""S(1257)""","""O[C]=CCC=C""","""AMChI=1/C5H7O/c1-3-5-4-2-6/h3-…","{200.0,3000.0,{5,7,1},0,1000.0,[3.130758, 0.040871, … 15.752088],[8.596625, 0.028732, … -12.719896],""nasa7""}"
"""C3Y12(175)""","""CC(C=O)=O""","""AMChI=1/C3H4O2/c1-3(5)2-4/h2H,…",0,0,"{4,null,null,null,null,3,null,2,null,null,null,null,null,null,null,null,null,null}",true,"""C3Y12(175)""","""O=C(C)C=O""","""AMChI=1/C3H4O2/c1-3(5)2-4/h2H,…","{200.0,3000.0,{3,4,2},0,1000.0,[4.375006, 0.01507, … 7.89432],[6.826185, 0.01914, … -6.856415],""nasa7""}"
"""S(1540)e""","""[H]/[C]=C/CC(=C)O""","""AMChI=1/C5H7O/c1-3-4-5(2)6/h1,…",1,0,"{7,null,null,null,null,5,null,1,null,null,null,null,null,null,null,null,null,null}",true,"""S(1540)""","""OC(CC=[CH])=C""","""AMChI=1/C5H7O/c1-3-4-5(2)6/h1,…","{200.0,3000.0,{5,7,1},0,1000.0,[3.935928, 0.03796, … 10.490996],[11.068416, 0.024864, … -26.964225],""nasa7""}"
…,…,…,…,…,…,…,…,…,…,…
"""C4H7O(419)rsR""","""[CH2][C@H]1[C@@H](C)O1""","""AMChI=1/C4H7O/c1-3-4(2)5-3/h3-…",1,0,"{7,null,null,null,null,4,null,1,null,null,null,null,null,null,null,null,null,null}",true,"""C4H7O(419)""","""CC1OC1[CH2]""","""AMChI=1/C4H7O/c1-3-4(2)5-3/h3-…","{200.0,3000.0,{4,7,1},0,1000.0,[3.725342, 0.023334, … 11.165119],[6.024382, 0.029382, … -3.22436],""nasa7""}"
"""S(1380)""","""O=C1C[CH]C1""","""AMChI=1/C4H5O/c5-4-2-1-3-4/h1H…",1,0,"{5,null,null,null,null,4,null,1,null,null,null,null,null,null,null,null,null,null}",true,"""S(1380)""","""O=C1C[CH]C1""","""AMChI=1/C4H5O/c5-4-2-1-3-4/h1H…","{200.0,3000.0,{4,5,1},0,1000.0,[3.061318, 0.015359, … 12.910777],[5.916623, 0.025186, … -5.44298],""nasa7""}"
"""C4H7O(394)e""","""C/C=C/C[O]""","""AMChI=1/C4H7O/c1-2-3-4-5/h2-3H…",1,0,"{7,null,null,null,null,4,null,1,null,null,null,null,null,null,null,null,null,null}",true,"""C4H7O(394)""","""[O]CC=CC""","""AMChI=1/C4H7O/c1-2-3-4-5/h2-3H…","{200.0,3000.0,{4,7,1},0,1000.0,[6.770893, 0.004285, … -0.95285],[5.032212, 0.031125, … 3.128329],""nasa7""}"
